In [ ]:
import subprocess
import sys
import os
import glob

# 1. Automatic installation of pandas if missing
try:
    import pandas as pd
except ImportError:
    print("Pandas not found. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd

# 2. Define the logic to combine files
def combine_csvs(output_filename="master_combined.csv"):
    # Find all CSVs in the current directory
    all_files = glob.glob("*.csv")

    # Filter out the master file if it already exists to avoid a loop
    all_files = [f for f in all_files if f != output_filename]

    if not all_files:
        print("No CSV files found in this directory.")
        return

    print(f"Combining {len(all_files)} files...")

    # Read and stack them
    df_list = [pd.read_csv(f) for f in all_files]
    master_df = pd.concat(df_list, axis=0, ignore_index=True)

    # Save the result
    master_df.to_csv(output_filename, index=False)
    print(f"Done! Master file saved as: {output_filename}")

if __name__ == "__main__":
    combine_csvs()

In [ ]:
import pandas as pd
import io

# Data provided by the user
data = """site_ID no_of_customers_connected
C070 1
C098 25
C102 406
C147 277
C188 104
C290 1
C306 64
C304 42
C359 98
C363 141
C615 154"""

# Parse the data into a DataFrame
customers_data = pd.read_csv(io.StringIO(data), sep=r'\s+')
print("Customer connection data loaded:")
print(customers_data.head())

# Load the /content/filtered_power_cut_data.csv generated by the previous cell
try:
    master_df = pd.read_csv("/content/filtered_power_cut_data.csv")
    print("\nLoaded '/content/filtered_power_cut_data.csv'. Current DataFrame head:")
    print(master_df.head())

    # Perform the merge operation
    # Assuming 'site_ID' is the common column
    merged_df = pd.merge(master_df, customers_data, on='site_ID', how='left')

    print("\nSuccessfully merged 'no_of_customers_connected' column.")
    print("Updated DataFrame head with new column:")
    print(merged_df.head())

    # Update the master_df reference to the merged DataFrame for subsequent operations
    master_df = merged_df

    # Optionally, save the updated DataFrame back to a new CSV or overwrite the old one
    # For now, let's just display and keep it in memory
    # master_df.to_csv("master_combined_with_customers.csv", index=False)
    # print("\nUpdated DataFrame saved as 'master_combined_with_customers.csv'")

except FileNotFoundError:
    print("\nError: 'master_combined.csv' not found. Please ensure the previous cell has been executed and the file was created.")
    print("Cannot proceed with merging without the base DataFrame.")
except Exception as e:
    print(f"\nAn error occurred during processing: {e}")

In [ ]:
master_df[master_df['site_ID'] == 'C070']

In [ ]:
master_df['power_cut_hours'] = master_df['power_cut_flag']/60

In [ ]:
master_df['CAIDI'] = master_df['power_cut_hours']/master_df['event_count']

In [ ]:
# master_df['CAIDI'].fillna(0, inplace = True)

In [ ]:
master_df['CAIDI']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure 'timestamp' is a datetime object
master_df['timestamp'] = pd.to_datetime(master_df['timestamp'])

# Filter for the desired time range (July 2025 to December 2025)
filtered_df = master_df[
    (master_df['timestamp'].dt.year == 2025) &
    (master_df['timestamp'].dt.month >= 7) &
    (master_df['timestamp'].dt.month <= 12)
].copy()

# Extract month as a string for better visualization on the x-axis
filtered_df['month'] = filtered_df['timestamp'].dt.strftime('%b %Y')

# Define a custom order for months
month_order = [
    'Jul 2025', 'Aug 2025', 'Sep 2025',
    'Oct 2025', 'Nov 2025', 'Dec 2025'
]

# Create a pivot table for the heatmap
heatmap_data = filtered_df.pivot_table(
    index='site_ID',
    columns='month',
    values='CAIDI',
    aggfunc='mean' # Aggregate if multiple entries for a site-month
).reindex(columns=month_order) # Reindex columns to ensure correct month order

# Create the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_data,
    cmap='YlOrRd', # Choose a color map
    annot=True, # Show the CAIDI values on the heatmap
    fmt=".2f", # Format annotation to one decimal place
    linewidths=.5 # Add lines between cells for better separation
)
plt.title('CAIDI (Jul - Dec 2025)')
plt.xlabel('Month')
plt.ylabel('Site ID')
plt.tight_layout()
plt.savefig('reliability_metrics_0725-1225.png')
plt.show()